In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 12_inference_from_upload_csv
# MAGIC 
# MAGIC Procesa archivos CSV subidos desde el dashboard y genera predicciones

# COMMAND ----------

# MAGIC %md
# MAGIC ## ⚙️ Configuración

# COMMAND ----------

import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from datetime import datetime, timedelta
import pickle
import json

# Rutas
UPLOAD_PATH = "/Volumes/olist/olist_uploads/inference/"  # Archivos subidos desde dashboard
MODELS_PATH = "/Volumes/olist/olist_gold/models/"
OUTPUT_PATH = "/Volumes/olist/olist_gold/inference_web/"

# Timestamp para esta ejecución
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

print("="*80)
print("🔮 INFERENCIA DESDE ARCHIVOS CSV CARGADOS")
print("="*80)
print(f"📅 Session ID: {TIMESTAMP}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 📥 Paso 1: Verificar archivos cargados

# COMMAND ----------

print("📂 PASO 1: VERIFICACIÓN DE ARCHIVOS\n" + "="*80 + "\n")

try:
    files = dbutils.fs.ls(UPLOAD_PATH)
    csv_files = [f for f in files if f.name.endswith('.csv')]
    
    print(f"✅ Archivos CSV encontrados: {len(csv_files)}\n")
    
    for f in csv_files:
        size_mb = f.size / (1024 * 1024)
        print(f"   • {f.name:45s} {size_mb:>8.2f} MB")
    
    if not csv_files:
        raise FileNotFoundError("No se encontraron archivos CSV")
    
    # Verificar archivos requeridos
    file_dict = {f.name.lower(): f.path for f in csv_files}
    
    required_files = [
        'olist_orders_dataset.csv',
        'olist_order_items_dataset.csv', 
        'olist_order_payments_dataset.csv',
        'olist_customers_dataset.csv'
    ]
    
    missing = []
    for req in required_files:
        if req not in file_dict:
            missing.append(req)
    
    if missing:
        print(f"\n❌ ERROR: Faltan archivos requeridos:")
        for m in missing:
            print(f"   • {m}")
        dbutils.notebook.exit(f"ERROR: Missing files: {missing}")
    
    print("\n✅ Todos los archivos requeridos presentes\n")
        
except Exception as e:
    print(f"\n❌ ERROR al listar archivos: {e}")
    dbutils.notebook.exit(f"ERROR: {e}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 📦 Paso 2: Cargar artefactos del modelo

# COMMAND ----------

print("📦 PASO 2: CARGA DE ARTEFACTOS\n" + "="*80 + "\n")

try:
    # Cargar metadata del modelo
    metadata_df = spark.read.format("delta").load(f"{MODELS_PATH}transformation_metadata/").toPandas()
    n_pca_expected = int(metadata_df['n_pca_components'].iloc[0])
    
    # Cargar features retenidas
    features_retained_df = spark.read.format("delta").load(f"{MODELS_PATH}features_retained/").toPandas()
    features_retained = features_retained_df.sort_values('order')['feature'].tolist()
    
    # Cargar modelo pickle
    with open(f"/dbfs{MODELS_PATH}best_model.pkl", 'rb') as f:
        model = pickle.load(f)
    
    # Cargar metadata JSON
    with open(f"/dbfs{MODELS_PATH}metadata.json", 'r') as f:
        model_metadata = json.load(f)
    
    print(f"✅ Artefactos cargados:")
    print(f"   • Modelo: {model_metadata['best_model']}")
    print(f"   • F1-Score: {model_metadata['best_f1_score']:.4f}")
    print(f"   • Componentes PCA: {n_pca_expected}")
    print(f"   • Features retenidas: {len(features_retained)}\n")
    
except Exception as e:
    print(f"❌ ERROR cargando artefactos: {e}")
    dbutils.notebook.exit(f"ERROR loading artifacts: {e}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 📊 Paso 3: Cargar y procesar CSVs

# COMMAND ----------

print("📥 PASO 3: CARGA DE CSVs\n" + "="*80 + "\n")

# 3.1 Cargar orders
print("1️⃣ Cargando orders...")
orders = spark.read.csv(
    file_dict['olist_orders_dataset.csv'],
    header=True,
    inferSchema=False
)

orders = orders \
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp")) \
    .withColumn("order_approved_at", F.to_timestamp("order_approved_at")) \
    .withColumn("order_delivered_carrier_date", F.to_timestamp("order_delivered_carrier_date")) \
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date")) \
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date"))

print(f"   ✅ {orders.count():,} órdenes\n")

# 3.2 Cargar order_items
print("2️⃣ Cargando order_items...")
order_items = spark.read.csv(
    file_dict['olist_order_items_dataset.csv'],
    header=True,
    inferSchema=False
)

order_items = order_items \
    .withColumn("order_item_id", F.col("order_item_id").cast("int")) \
    .withColumn("price", F.col("price").cast("double")) \
    .withColumn("freight_value", F.col("freight_value").cast("double"))

print(f"   ✅ {order_items.count():,} items\n")

# 3.3 Cargar payments
print("3️⃣ Cargando payments...")
payments = spark.read.csv(
    file_dict['olist_order_payments_dataset.csv'],
    header=True,
    inferSchema=False
)

payments = payments \
    .withColumn("payment_sequential", F.col("payment_sequential").cast("int")) \
    .withColumn("payment_installments", F.col("payment_installments").cast("int")) \
    .withColumn("payment_value", F.col("payment_value").cast("double"))

print(f"   ✅ {payments.count():,} pagos\n")

# 3.4 Cargar customers
print("4️⃣ Cargando customers...")
customers = spark.read.csv(
    file_dict['olist_customers_dataset.csv'],
    header=True,
    inferSchema=False
)

print(f"   ✅ {customers.count():,} clientes\n")

# 3.5 Cargar reviews (opcional)
if 'olist_order_reviews_dataset.csv' in file_dict:
    print("5️⃣ Cargando reviews...")
    reviews = spark.read.csv(
        file_dict['olist_order_reviews_dataset.csv'],
        header=True,
        inferSchema=False
    )
    
    reviews = reviews \
        .withColumn("review_creation_date", F.to_timestamp("review_creation_date")) \
        .withColumn("review_answer_timestamp", F.to_timestamp("review_answer_timestamp")) \
        .withColumn("review_score", F.expr("try_cast(review_score as int)")) \
        .filter(F.col("review_score").isNotNull())
    
    print(f"   ✅ {reviews.count():,} reviews\n")
    has_reviews = True
else:
    print("⚠️  Reviews no disponibles (opcional)\n")
    has_reviews = False

# COMMAND ----------

# MAGIC %md
# MAGIC ## 🔧 Paso 4: Agregar y combinar datos

# COMMAND ----------

print("🔧 PASO 4: AGREGACIÓN DE DATOS\n" + "="*80 + "\n")

# 4.1 Agregar items por orden
print("📦 Agregando items...")
items_agg = order_items.groupBy("order_id").agg(
    F.count("*").alias("items_count"),
    F.countDistinct("product_id").alias("distinct_products"),
    F.sum("price").alias("sum_price"),
    F.sum("freight_value").alias("sum_freight")
)

# 4.2 Agregar payments por orden
print("💳 Agregando pagos...")
payments_agg = payments.groupBy("order_id").agg(
    F.sum("payment_value").alias("payment_sum"),
    F.avg("payment_installments").alias("avg_installments"),
    F.countDistinct("payment_type").alias("n_payment_types")
)

# 4.3 Agregar reviews si disponibles
if has_reviews:
    print("⭐ Agregando reviews...")
    reviews_agg = reviews.groupBy("order_id").agg(
        F.avg("review_score").alias("avg_review_score"),
        F.min("review_score").alias("min_review_score"),
        F.max("review_score").alias("max_review_score")
    )
else:
    reviews_agg = None

# 4.4 Combinar todo
print("🔗 Combinando fuentes...")
orders_full = orders \
    .join(items_agg, "order_id", "left") \
    .join(payments_agg, "order_id", "left")

if has_reviews:
    orders_full = orders_full.join(reviews_agg, "order_id", "left")

orders_full = orders_full.fillna(0)

print(f"✅ Orders combinados: {orders_full.count():,}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 📅 Paso 5: Determinar periodo

# COMMAND ----------

print("📅 PASO 5: ANÁLISIS DE PERIODO\n" + "="*80 + "\n")

# Obtener fecha máxima
date_stats = orders_full.select(
    F.min('order_purchase_timestamp').alias('min_date'),
    F.max('order_purchase_timestamp').alias('max_date')
).collect()[0]

min_date = date_stats['min_date']
max_date = date_stats['max_date']

print(f"📊 Rango de fechas:")
print(f"   • Mínima: {min_date}")
print(f"   • Máxima: {max_date}\n")

# Usar últimos 60 días
max_dt = max_date if isinstance(max_date, datetime) else datetime.strptime(str(max_date), '%Y-%m-%d %H:%M:%S')
PERIODO_DIAS = 60
start_dt = max_dt - timedelta(days=PERIODO_DIAS)

START_DATE = start_dt.strftime('%Y-%m-%d %H:%M:%S')
END_DATE = max_dt.strftime('%Y-%m-%d %H:%M:%S')
CUTOFF_DATE = END_DATE

print(f"🗓️  Periodo seleccionado ({PERIODO_DIAS} días):")
print(f"   • Inicio: {START_DATE}")
print(f"   • Fin:    {END_DATE}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 🔍 Paso 6: Filtrar periodo

# COMMAND ----------

print("🔍 PASO 6: FILTRADO\n" + "="*80 + "\n")

orders_production = orders_full.filter(
    (F.col("order_purchase_timestamp") >= F.lit(START_DATE)) &
    (F.col("order_purchase_timestamp") <= F.lit(END_DATE)) &
    (F.col("order_status") != "canceled") &
    (F.col("customer_id").isNotNull())
)

n_orders = orders_production.count()
n_customers = orders_production.select("customer_id").distinct().count()

print(f"📊 Órdenes filtradas: {n_orders:,}")
print(f"👥 Clientes únicos: {n_customers:,}\n")

if n_orders == 0:
    print("❌ No hay órdenes en el periodo")
    dbutils.notebook.exit("ERROR: No orders in period")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 🎯 Paso 7: Generar features

# COMMAND ----------

print("🎯 PASO 7: GENERACIÓN DE FEATURES\n" + "="*80 + "\n")

features = orders_production.groupBy("customer_id").agg(
    # RFM
    F.datediff(F.lit(CUTOFF_DATE), F.max("order_purchase_timestamp")).alias("recency"),
    F.count("order_id").alias("frequency"),
    F.sum("payment_sum").alias("monetary"),
    
    # Ticket
    F.avg("payment_sum").alias("avg_ticket"),
    F.max("payment_sum").alias("max_ticket"),
    F.min("payment_sum").alias("min_ticket"),
    F.stddev("payment_sum").alias("std_ticket"),
    
    # Items
    F.avg("items_count").alias("avg_items_per_order"),
    F.max("items_count").alias("max_items_per_order"),
    F.sum("items_count").alias("total_items"),
    F.avg("distinct_products").alias("avg_distinct_products"),
    F.sum("distinct_products").alias("total_distinct_products"),
    
    # Precios
    F.avg("sum_price").alias("avg_price"),
    F.sum("sum_price").alias("total_price"),
    F.avg("sum_freight").alias("avg_freight"),
    F.sum("sum_freight").alias("total_freight"),
    
    # Pagos
    F.avg("avg_installments").alias("avg_installments"),
    F.max("avg_installments").alias("max_installments"),
    F.avg("n_payment_types").alias("avg_payment_types"),
    
    # Reviews
    F.avg("avg_review_score").alias("avg_review_score"),
    F.min("min_review_score").alias("min_review_score"),
    F.max("max_review_score").alias("max_review_score"),
    F.count(F.when(F.col("avg_review_score").isNotNull(), 1)).alias("orders_with_review"),
    
    # Temporales
    F.min("order_purchase_timestamp").alias("first_purchase"),
    F.max("order_purchase_timestamp").alias("last_purchase"),
    F.datediff(F.max("order_purchase_timestamp"), F.min("order_purchase_timestamp")).alias("customer_lifetime_days"),
    
    # Entrega
    F.avg(F.datediff("order_delivered_customer_date", "order_purchase_timestamp")).alias("avg_delivery_days"),
    F.max(F.datediff("order_delivered_customer_date", "order_purchase_timestamp")).alias("max_delivery_days"),
    F.avg(F.datediff("order_delivered_customer_date", "order_estimated_delivery_date")).alias("avg_delay_days"),
    F.count(F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), 1)).alias("delayed_orders"),
    
    # Status
    F.count(F.when(F.col("order_status") == "delivered", 1)).alias("delivered_orders"),
    F.count(F.when(F.col("order_status") == "shipped", 1)).alias("shipped_orders")
)

# Features temporales
features = features \
    .withColumn("first_purchase_month", F.month("first_purchase")) \
    .withColumn("first_purchase_day", F.dayofmonth("first_purchase")) \
    .withColumn("first_purchase_dow", F.dayofweek("first_purchase")) \
    .withColumn("last_purchase_month", F.month("last_purchase")) \
    .withColumn("last_purchase_day", F.dayofmonth("last_purchase")) \
    .withColumn("last_purchase_dow", F.dayofweek("last_purchase"))

# Quitar timestamps
features = features.drop("first_purchase", "last_purchase")

# Features derivadas
features = features \
    .withColumn("ticket_range", F.col("max_ticket") - F.col("min_ticket")) \
    .withColumn("freight_ratio", F.col("total_freight") / (F.col("total_price") + 1)) \
    .withColumn("items_per_product", F.col("total_items") / (F.col("total_distinct_products") + 1)) \
    .withColumn("review_consistency", F.col("max_review_score") - F.col("min_review_score")) \
    .withColumn("orders_per_day", F.col("frequency") / (F.col("customer_lifetime_days") + 1)) \
    .fillna(0)

print(f"✅ Features generadas: {len(features.columns)}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 🔄 Paso 8: Aplicar PCA

# COMMAND ----------

print("🔄 PASO 8: TRANSFORMACIÓN PCA\n" + "="*80 + "\n")

# Convertir a pandas
features_pd = features.toPandas()
customer_ids = features_pd['customer_id'].copy()

# Seleccionar solo features numéricas (excluir customer_id)
feature_cols = [c for c in features_pd.columns if c != 'customer_id']
X = features_pd[feature_cols].fillna(0)

print(f"📊 Features antes de PCA: {X.shape}")

# Filtrar features retenidas
available_features = [f for f in features_retained if f in X.columns]
X_retained = X[available_features].copy()

print(f"📊 Features retenidas: {len(available_features)}")

# Cargar scaler y PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_retained)

pca = PCA(n_components=n_pca_expected)
X_pca = pca.fit_transform(X_scaled)

print(f"📊 Features después de PCA: {X_pca.shape}")
print(f"✅ Varianza explicada: {pca.explained_variance_ratio_.sum()*100:.2f}%\n")

# Crear DataFrame con componentes PCA
pca_cols = [f'pca_{i+1}' for i in range(n_pca_expected)]
X_pca_df = pd.DataFrame(X_pca, columns=pca_cols)
X_pca_df['customer_id'] = customer_ids.values

# COMMAND ----------

# MAGIC %md
# MAGIC ## 🔮 Paso 9: Generar predicciones

# COMMAND ----------

print("🔮 PASO 9: PREDICCIONES\n" + "="*80 + "\n")

# Preparar datos para modelo
X_inference = X_pca_df[pca_cols].copy()

# Predicciones
predictions = model.predict(X_inference)
probabilities = model.predict_proba(X_inference)[:, 1]

print(f"✅ Predicciones generadas: {len(predictions):,}\n")

# Crear DataFrame de resultados
results_df = X_pca_df[['customer_id']].copy()
results_df['is_premium_predicted'] = predictions
results_df['premium_probability'] = probabilities
results_df['premium_label'] = results_df['is_premium_predicted'].map({
    0: 'REGULAR',
    1: 'PREMIUM'
})
results_df['confidence'] = results_df['premium_probability'].apply(
    lambda x: 'HIGH' if x > 0.8 or x < 0.2 
              else 'MEDIUM' if x > 0.6 or x < 0.4 
              else 'LOW'
)

# Metadata
results_df['prediction_date'] = datetime.now()
results_df['model_name'] = model_metadata['best_model']
results_df['session_id'] = TIMESTAMP

# Agregar features RFM originales para contexto
rfm_features = features_pd[['customer_id', 'recency', 'frequency', 'monetary']].copy()
results_df = results_df.merge(rfm_features, on='customer_id', how='left')

# Estadísticas
premium_count = predictions.sum()
premium_pct = (premium_count / len(predictions)) * 100

print(f"📈 Estadísticas:")
print(f"   • Total clientes: {len(results_df):,}")
print(f"   • PREMIUM: {premium_count:,} ({premium_pct:.1f}%)")
print(f"   • REGULAR: {len(results_df) - premium_count:,}")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 💾 Paso 10: Guardar resultados

# COMMAND ----------

print("💾 PASO 10: GUARDAR RESULTADOS\n" + "="*80 + "\n")

# Crear volume si no existe
try:
    spark.sql("CREATE VOLUME IF NOT EXISTS olist.olist_gold.inference_web")
except:
    pass

# Guardar como Delta
output_table = f"{OUTPUT_PATH}predictions_{TIMESTAMP}/"

spark.createDataFrame(results_df).write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(output_table)

print(f"✅ Delta guardado: {output_table}")

# Guardar como CSV
output_csv = f"{OUTPUT_PATH}predictions_{TIMESTAMP}.csv"
results_df.to_csv(f"/dbfs{output_csv}", index=False)

print(f"✅ CSV guardado: {output_csv}")

# Guardar como tabla para dashboard
spark.createDataFrame(results_df).write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("olist.olist_gold.inference_web_latest")

print(f"✅ Tabla creada: olist.olist_gold.inference_web_latest")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 🎉 Resumen Final

# COMMAND ----------

print("="*80)
print("🎉 INFERENCIA COMPLETADA")
print("="*80)
print(f"\n📊 Resumen:")
print(f"   • Session ID: {TIMESTAMP}")
print(f"   • Modelo: {model_metadata['best_model']}")
print(f"   • Clientes procesados: {len(results_df):,}")
print(f"   • PREMIUM: {premium_count:,} ({premium_pct:.1f}%)")
print(f"   • REGULAR: {len(results_df) - premium_count:,}")
print()
print(f"💾 Resultados:")
print(f"   • Delta: {output_table}")
print(f"   • CSV: {output_csv}")
print(f"   • Tabla: olist.olist_gold.inference_web_latest")
print()
print("="*80)

# Retornar resultado
result = {
    "status": "SUCCESS",
    "timestamp": TIMESTAMP,
    "total_customers": len(results_df),
    "premium_customers": int(premium_count),
    "premium_percentage": float(premium_pct),
    "output_table": output_table,
    "output_csv": output_csv
}

dbutils.notebook.exit(json.dumps(result))

In [0]:
# Databricks notebook source
# Databricks notebook source
# MAGIC %md
# MAGIC # 12_inference_from_upload_csv
# MAGIC 
# MAGIC Procesa archivos CSV subidos desde el dashboard y genera predicciones

# COMMAND ----------

# MAGIC %md
# MAGIC ## ⚙️ Configuración

# COMMAND ----------

import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from datetime import datetime, timedelta
import pickle
import json

# Rutas
UPLOAD_PATH = "/Volumes/olist/olist_uploads/inference/"  # Archivos subidos desde dashboard
MODELS_PATH = "/Volumes/olist/olist_gold/models/"
OUTPUT_PATH = "/Volumes/olist/olist_gold/inference_web/"

# Timestamp para esta ejecución
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

print("="*80)
print("🔮 INFERENCIA DESDE ARCHIVOS CSV CARGADOS")
print("="*80)
print(f"📅 Session ID: {TIMESTAMP}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 📥 Paso 1: Verificar archivos cargados

# COMMAND ----------

print("📂 PASO 1: VERIFICACIÓN DE ARCHIVOS\n" + "="*80 + "\n")

try:
    files = dbutils.fs.ls(UPLOAD_PATH)
    csv_files = [f for f in files if f.name.endswith('.csv')]
    
    print(f"✅ Archivos CSV encontrados: {len(csv_files)}\n")
    
    for f in csv_files:
        size_mb = f.size / (1024 * 1024)
        print(f"   • {f.name:45s} {size_mb:>8.2f} MB")
    
    if not csv_files:
        raise FileNotFoundError("No se encontraron archivos CSV")
    
    # Verificar archivos requeridos
    file_dict = {f.name.lower(): f.path for f in csv_files}
    
    required_files = [
        'olist_orders_dataset.csv',
        'olist_order_items_dataset.csv', 
        'olist_order_payments_dataset.csv',
        'olist_customers_dataset.csv'
    ]
    
    missing = []
    for req in required_files:
        if req not in file_dict:
            missing.append(req)
    
    if missing:
        print(f"\n❌ ERROR: Faltan archivos requeridos:")
        for m in missing:
            print(f"   • {m}")
        dbutils.notebook.exit(f"ERROR: Missing files: {missing}")
    
    print("\n✅ Todos los archivos requeridos presentes\n")
        
except Exception as e:
    print(f"\n❌ ERROR al listar archivos: {e}")
    dbutils.notebook.exit(f"ERROR: {e}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 📦 Paso 2: Cargar artefactos del modelo (DESDE MLFLOW)

# COMMAND ----------

print("📦 PASO 2: CARGA DE ARTEFACTOS (MLFLOW)\n" + "="*80 + "\n")

import mlflow
import mlflow.sklearn

# ⚠️ CONFIGURAR: Run ID del modelo en MLflow
MLFLOW_RUN_ID = "aa1ead44843d4b70bc9abbf9e9ec479c"  # ← Cambiar si es necesario

try:
    # 1. Cargar transformaciones PCA
    print("📊 Cargando transformaciones PCA...\n")
    
    metadata_df = spark.read.format("delta") \
        .load(f"{MODELS_PATH}transformation_metadata/") \
        .toPandas()
    
    n_pca_expected = int(metadata_df['n_pca_components'].iloc[0])
    
    features_retained_df = spark.read.format("delta") \
        .load(f"{MODELS_PATH}features_retained/") \
        .toPandas()
    
    features_retained = features_retained_df.sort_values('order')['feature'].tolist()
    
    print(f"✅ Transformaciones PCA cargadas:")
    print(f"   • Componentes PCA: {n_pca_expected}")
    print(f"   • Features retenidas: {len(features_retained)}\n")
    
except Exception as e:
    print(f"❌ ERROR cargando transformaciones PCA: {e}")
    print("\n💡 SOLUCIÓN:")
    print("   Ejecuta notebook: 06_pca_transformation")
    dbutils.notebook.exit(f"ERROR loading PCA: {e}")

try:
    # 2. Cargar modelo desde MLflow
    print("🤖 Cargando modelo desde MLflow...\n")
    print(f"   Run ID: {MLFLOW_RUN_ID}\n")
    
    model = mlflow.sklearn.load_model(f"runs:/{MLFLOW_RUN_ID}/model")
    
    print("✅ Modelo cargado desde MLflow\n")
    
    # Obtener información del run
    client = mlflow.tracking.MlflowClient()
    run = client.get_run(MLFLOW_RUN_ID)
    
    # Extraer métricas del run
    metrics = run.data.metrics
    params = run.data.params
    
    model_metadata = {
        "best_model": params.get('model_name', run.data.tags.get('mlflow.runName', 'Unknown')),
        "best_f1_score": metrics.get('f1_score', metrics.get('test_f1', 0.0)),
        "accuracy": metrics.get('accuracy', metrics.get('test_accuracy', 0.0)),
        "run_id": MLFLOW_RUN_ID
    }
    
    print(f"📊 Información del modelo:")
    print(f"   • Modelo: {model_metadata['best_model']}")
    print(f"   • F1-Score: {model_metadata['best_f1_score']:.4f}")
    print(f"   • Accuracy: {model_metadata.get('accuracy', 0.0):.4f}")
    print(f"   • Run ID: {MLFLOW_RUN_ID}\n")
    
except Exception as e:
    print(f"❌ ERROR cargando modelo desde MLflow: {e}")
    print(f"\n💡 SOLUCIÓN:")
    print(f"   1. Verifica que el Run ID es correcto: {MLFLOW_RUN_ID}")
    print(f"   2. Ve a MLflow → Experiments → Busca el run")
    print(f"   3. Verifica que el modelo existe en ese run")
    print(f"   4. Comando para listar runs:")
    print(f"      mlflow.search_runs(experiment_ids=['...'])")
    dbutils.notebook.exit(f"ERROR loading model from MLflow: {e}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 📊 Paso 3: Cargar y procesar CSVs

# COMMAND ----------

print("📥 PASO 3: CARGA DE CSVs\n" + "="*80 + "\n")

# 3.1 Cargar orders
print("1️⃣ Cargando orders...")
orders = spark.read.csv(
    file_dict['olist_orders_dataset.csv'],
    header=True,
    inferSchema=False
)

orders = orders \
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp")) \
    .withColumn("order_approved_at", F.to_timestamp("order_approved_at")) \
    .withColumn("order_delivered_carrier_date", F.to_timestamp("order_delivered_carrier_date")) \
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date")) \
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date"))

print(f"   ✅ {orders.count():,} órdenes\n")

# 3.2 Cargar order_items
print("2️⃣ Cargando order_items...")
order_items = spark.read.csv(
    file_dict['olist_order_items_dataset.csv'],
    header=True,
    inferSchema=False
)

order_items = order_items \
    .withColumn("order_item_id", F.col("order_item_id").cast("int")) \
    .withColumn("price", F.col("price").cast("double")) \
    .withColumn("freight_value", F.col("freight_value").cast("double"))

print(f"   ✅ {order_items.count():,} items\n")

# 3.3 Cargar payments
print("3️⃣ Cargando payments...")
payments = spark.read.csv(
    file_dict['olist_order_payments_dataset.csv'],
    header=True,
    inferSchema=False
)

payments = payments \
    .withColumn("payment_sequential", F.col("payment_sequential").cast("int")) \
    .withColumn("payment_installments", F.col("payment_installments").cast("int")) \
    .withColumn("payment_value", F.col("payment_value").cast("double"))

print(f"   ✅ {payments.count():,} pagos\n")

# 3.4 Cargar customers
print("4️⃣ Cargando customers...")
customers = spark.read.csv(
    file_dict['olist_customers_dataset.csv'],
    header=True,
    inferSchema=False
)

print(f"   ✅ {customers.count():,} clientes\n")

# 3.5 Cargar reviews (opcional)
if 'olist_order_reviews_dataset.csv' in file_dict:
    print("5️⃣ Cargando reviews...")
    reviews = spark.read.csv(
        file_dict['olist_order_reviews_dataset.csv'],
        header=True,
        inferSchema=False
    )
    
    reviews = reviews \
        .withColumn("review_creation_date", F.to_timestamp("review_creation_date")) \
        .withColumn("review_answer_timestamp", F.to_timestamp("review_answer_timestamp")) \
        .withColumn("review_score", F.expr("try_cast(review_score as int)")) \
        .filter(F.col("review_score").isNotNull())
    
    print(f"   ✅ {reviews.count():,} reviews\n")
    has_reviews = True
else:
    print("⚠️  Reviews no disponibles (opcional)\n")
    has_reviews = False

# COMMAND ----------

# MAGIC %md
# MAGIC ## 🔧 Paso 4: Agregar y combinar datos

# COMMAND ----------

print("🔧 PASO 4: AGREGACIÓN DE DATOS\n" + "="*80 + "\n")

# 4.1 Agregar items por orden
print("📦 Agregando items...")
items_agg = order_items.groupBy("order_id").agg(
    F.count("*").alias("items_count"),
    F.countDistinct("product_id").alias("distinct_products"),
    F.sum("price").alias("sum_price"),
    F.sum("freight_value").alias("sum_freight")
)

# 4.2 Agregar payments por orden
print("💳 Agregando pagos...")
payments_agg = payments.groupBy("order_id").agg(
    F.sum("payment_value").alias("payment_sum"),
    F.avg("payment_installments").alias("avg_installments"),
    F.countDistinct("payment_type").alias("n_payment_types")
)

# 4.3 Agregar reviews si disponibles
if has_reviews:
    print("⭐ Agregando reviews...")
    reviews_agg = reviews.groupBy("order_id").agg(
        F.avg("review_score").alias("avg_review_score"),
        F.min("review_score").alias("min_review_score"),
        F.max("review_score").alias("max_review_score")
    )
else:
    reviews_agg = None

# 4.4 Combinar todo
print("🔗 Combinando fuentes...")
orders_full = orders \
    .join(items_agg, "order_id", "left") \
    .join(payments_agg, "order_id", "left")

if has_reviews:
    orders_full = orders_full.join(reviews_agg, "order_id", "left")

orders_full = orders_full.fillna(0)

print(f"✅ Orders combinados: {orders_full.count():,}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 📅 Paso 5: Determinar periodo

# COMMAND ----------

print("📅 PASO 5: ANÁLISIS DE PERIODO\n" + "="*80 + "\n")

# Obtener fecha máxima
date_stats = orders_full.select(
    F.min('order_purchase_timestamp').alias('min_date'),
    F.max('order_purchase_timestamp').alias('max_date')
).collect()[0]

min_date = date_stats['min_date']
max_date = date_stats['max_date']

print(f"📊 Rango de fechas:")
print(f"   • Mínima: {min_date}")
print(f"   • Máxima: {max_date}\n")

# Usar últimos 60 días
max_dt = max_date if isinstance(max_date, datetime) else datetime.strptime(str(max_date), '%Y-%m-%d %H:%M:%S')
PERIODO_DIAS = 60
start_dt = max_dt - timedelta(days=PERIODO_DIAS)

START_DATE = start_dt.strftime('%Y-%m-%d %H:%M:%S')
END_DATE = max_dt.strftime('%Y-%m-%d %H:%M:%S')
CUTOFF_DATE = END_DATE

print(f"🗓️  Periodo seleccionado ({PERIODO_DIAS} días):")
print(f"   • Inicio: {START_DATE}")
print(f"   • Fin:    {END_DATE}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 🔍 Paso 6: Filtrar periodo

# COMMAND ----------

print("🔍 PASO 6: FILTRADO\n" + "="*80 + "\n")

orders_production = orders_full.filter(
    (F.col("order_purchase_timestamp") >= F.lit(START_DATE)) &
    (F.col("order_purchase_timestamp") <= F.lit(END_DATE)) &
    (F.col("order_status") != "canceled") &
    (F.col("customer_id").isNotNull())
)

n_orders = orders_production.count()
n_customers = orders_production.select("customer_id").distinct().count()

print(f"📊 Órdenes filtradas: {n_orders:,}")
print(f"👥 Clientes únicos: {n_customers:,}\n")

if n_orders == 0:
    print("❌ No hay órdenes en el periodo")
    dbutils.notebook.exit("ERROR: No orders in period")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 🎯 Paso 7: Generar features

# COMMAND ----------

print("🎯 PASO 7: GENERACIÓN DE FEATURES\n" + "="*80 + "\n")

features = orders_production.groupBy("customer_id").agg(
    # RFM
    F.datediff(F.lit(CUTOFF_DATE), F.max("order_purchase_timestamp")).alias("recency"),
    F.count("order_id").alias("frequency"),
    F.sum("payment_sum").alias("monetary"),
    
    # Ticket
    F.avg("payment_sum").alias("avg_ticket"),
    F.max("payment_sum").alias("max_ticket"),
    F.min("payment_sum").alias("min_ticket"),
    F.stddev("payment_sum").alias("std_ticket"),
    
    # Items
    F.avg("items_count").alias("avg_items_per_order"),
    F.max("items_count").alias("max_items_per_order"),
    F.sum("items_count").alias("total_items"),
    F.avg("distinct_products").alias("avg_distinct_products"),
    F.sum("distinct_products").alias("total_distinct_products"),
    
    # Precios
    F.avg("sum_price").alias("avg_price"),
    F.sum("sum_price").alias("total_price"),
    F.avg("sum_freight").alias("avg_freight"),
    F.sum("sum_freight").alias("total_freight"),
    
    # Pagos
    F.avg("avg_installments").alias("avg_installments"),
    F.max("avg_installments").alias("max_installments"),
    F.avg("n_payment_types").alias("avg_payment_types"),
    
    # Reviews
    F.avg("avg_review_score").alias("avg_review_score"),
    F.min("min_review_score").alias("min_review_score"),
    F.max("max_review_score").alias("max_review_score"),
    F.count(F.when(F.col("avg_review_score").isNotNull(), 1)).alias("orders_with_review"),
    
    # Temporales
    F.min("order_purchase_timestamp").alias("first_purchase"),
    F.max("order_purchase_timestamp").alias("last_purchase"),
    F.datediff(F.max("order_purchase_timestamp"), F.min("order_purchase_timestamp")).alias("customer_lifetime_days"),
    
    # Entrega
    F.avg(F.datediff("order_delivered_customer_date", "order_purchase_timestamp")).alias("avg_delivery_days"),
    F.max(F.datediff("order_delivered_customer_date", "order_purchase_timestamp")).alias("max_delivery_days"),
    F.avg(F.datediff("order_delivered_customer_date", "order_estimated_delivery_date")).alias("avg_delay_days"),
    F.count(F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), 1)).alias("delayed_orders"),
    
    # Status
    F.count(F.when(F.col("order_status") == "delivered", 1)).alias("delivered_orders"),
    F.count(F.when(F.col("order_status") == "shipped", 1)).alias("shipped_orders")
)

# Features temporales
features = features \
    .withColumn("first_purchase_month", F.month("first_purchase")) \
    .withColumn("first_purchase_day", F.dayofmonth("first_purchase")) \
    .withColumn("first_purchase_dow", F.dayofweek("first_purchase")) \
    .withColumn("last_purchase_month", F.month("last_purchase")) \
    .withColumn("last_purchase_day", F.dayofmonth("last_purchase")) \
    .withColumn("last_purchase_dow", F.dayofweek("last_purchase"))

# Quitar timestamps
features = features.drop("first_purchase", "last_purchase")

# Features derivadas
features = features \
    .withColumn("ticket_range", F.col("max_ticket") - F.col("min_ticket")) \
    .withColumn("freight_ratio", F.col("total_freight") / (F.col("total_price") + 1)) \
    .withColumn("items_per_product", F.col("total_items") / (F.col("total_distinct_products") + 1)) \
    .withColumn("review_consistency", F.col("max_review_score") - F.col("min_review_score")) \
    .withColumn("orders_per_day", F.col("frequency") / (F.col("customer_lifetime_days") + 1)) \
    .fillna(0)

print(f"✅ Features generadas: {len(features.columns)}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 🔄 Paso 8: Aplicar PCA

# COMMAND ----------

print("🔄 PASO 8: TRANSFORMACIÓN PCA\n" + "="*80 + "\n")

# Convertir a pandas
features_pd = features.toPandas()
customer_ids = features_pd['customer_id'].copy()

# Seleccionar solo features numéricas (excluir customer_id)
feature_cols = [c for c in features_pd.columns if c != 'customer_id']
X = features_pd[feature_cols].fillna(0)

print(f"📊 Features antes de PCA: {X.shape}")

# Filtrar features retenidas
available_features = [f for f in features_retained if f in X.columns]
X_retained = X[available_features].copy()

print(f"📊 Features retenidas: {len(available_features)}")

# Cargar scaler y PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_retained)

pca = PCA(n_components=n_pca_expected)
X_pca = pca.fit_transform(X_scaled)

print(f"📊 Features después de PCA: {X_pca.shape}")
print(f"✅ Varianza explicada: {pca.explained_variance_ratio_.sum()*100:.2f}%\n")

# Crear DataFrame con componentes PCA
pca_cols = [f'pca_{i+1}' for i in range(n_pca_expected)]
X_pca_df = pd.DataFrame(X_pca, columns=pca_cols)
X_pca_df['customer_id'] = customer_ids.values

# COMMAND ----------

# MAGIC %md
# MAGIC ## 🔮 Paso 9: Generar predicciones

# COMMAND ----------

print("🔮 PASO 9: PREDICCIONES\n" + "="*80 + "\n")

# Preparar datos para modelo
X_inference = X_pca_df[pca_cols].copy()

# Predicciones
predictions = model.predict(X_inference)
probabilities = model.predict_proba(X_inference)[:, 1]

print(f"✅ Predicciones generadas: {len(predictions):,}\n")

# Crear DataFrame de resultados
results_df = X_pca_df[['customer_id']].copy()
results_df['is_premium_predicted'] = predictions
results_df['premium_probability'] = probabilities
results_df['premium_label'] = results_df['is_premium_predicted'].map({
    0: 'REGULAR',
    1: 'PREMIUM'
})
results_df['confidence'] = results_df['premium_probability'].apply(
    lambda x: 'HIGH' if x > 0.8 or x < 0.2 
              else 'MEDIUM' if x > 0.6 or x < 0.4 
              else 'LOW'
)

# Metadata
results_df['prediction_date'] = datetime.now()
results_df['model_name'] = model_metadata['best_model']
results_df['session_id'] = TIMESTAMP

# Agregar features RFM originales para contexto
rfm_features = features_pd[['customer_id', 'recency', 'frequency', 'monetary']].copy()
results_df = results_df.merge(rfm_features, on='customer_id', how='left')

# Estadísticas
premium_count = predictions.sum()
premium_pct = (premium_count / len(predictions)) * 100

print(f"📈 Estadísticas:")
print(f"   • Total clientes: {len(results_df):,}")
print(f"   • PREMIUM: {premium_count:,} ({premium_pct:.1f}%)")
print(f"   • REGULAR: {len(results_df) - premium_count:,}")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 💾 Paso 10: Guardar resultados

# COMMAND ----------

print("💾 PASO 10: GUARDAR RESULTADOS\n" + "="*80 + "\n")

# Crear volume si no existe
# Crear volume si no existe
try:
    spark.sql("CREATE VOLUME IF NOT EXISTS olist.olist_gold.inference_web")
    print("✅ Volume verificado")
except Exception as e:
    print(f"⚠️ No se pudo crear volume: {e}")
    print("   Usando ruta alternativa...")

# Guardar como Delta
output_table = f"{OUTPUT_PATH}predictions_{TIMESTAMP}/"

spark.createDataFrame(results_df).write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(output_table)

print(f"✅ Delta guardado: {output_table}")

# Guardar como CSV usando Spark (más robusto)
output_csv_dir = f"{OUTPUT_PATH}predictions_{TIMESTAMP}_csv/"

spark.createDataFrame(results_df).coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save(output_csv_dir)

print(f"✅ CSV guardado en: {output_csv_dir}")###################################


print(f"✅ CSV guardado: {output_csv}")

# Guardar como tabla para dashboard
spark.createDataFrame(results_df).write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("olist.olist_gold.inference_web_latest")

print(f"✅ Tabla creada: olist.olist_gold.inference_web_latest")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 🎉 Resumen Final

# COMMAND ----------

print("="*80)
print("🎉 INFERENCIA COMPLETADA")
print("="*80)
print(f"\n📊 Resumen:")
print(f"   • Session ID: {TIMESTAMP}")
print(f"   • Modelo: {model_metadata['best_model']}")
print(f"   • Clientes procesados: {len(results_df):,}")
print(f"   • PREMIUM: {premium_count:,} ({premium_pct:.1f}%)")
print(f"   • REGULAR: {len(results_df) - premium_count:,}")
print()
print(f"💾 Resultados:")
print(f"   • Delta: {output_table}")
print(f"   • CSV: {output_csv}")
print(f"   • Tabla: olist.olist_gold.inference_web_latest")
print()
print("="*80)

# Retornar resultado
result = {
    "status": "SUCCESS",
    "timestamp": TIMESTAMP,
    "total_customers": len(results_df),
    "premium_customers": int(premium_count),
    "premium_percentage": float(premium_pct),
    "output_table": output_table,
    "output_csv": output_csv
}

dbutils.notebook.exit(json.dumps(result))